# Module 1.4: Baseline Drift & Industrial Sensor Calibration
### 🇵🇱 Dryf Linii Bazowej i Kalibracja Czujników

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/01_04_baseline_calibration.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Remove slow thermal trends, calculate tare offsets, and correct baseline wandering on strain gauges and thermocouples.  
> 🇵🇱 **Streszczenie:** Usuwaj powolne trendy termiczne, wyliczaj offset tarowania i koryguj dryf bazowy tensometrów.

- **Estimated Time:** 20 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Sampling Frequency & Nyquist Criterion**: $F_s \ge 2 \cdot F_{max}$ is required to avoid frequency aliasing.
2. **Noise Sources**: High-voltage VFD drives and welding arcs induce electromagnetic high-frequency noise into sensor wiring.
3. **Phase Delay vs Smoothing**: Larger filter windows eliminate noise but introduce group delay, slowing safety shutdown reactions.


In [ ]:
# 1. Simulate High-Speed Factory Telemetry (Vibration & Spikes)
np.random.seed(42)
sampling_rate = 1000  # Hz
duration = 2.0        # seconds
t = np.linspace(0, duration, int(sampling_rate * duration), endpoint=False)

# Fundamental mechanical rotation (30 Hz ~ 1800 RPM) + 90 Hz harmonic
clean_signal = 2.5 * np.sin(2 * np.pi * 30 * t) + 1.2 * np.sin(2 * np.pi * 90 * t)

# Add electromagnetic sensor noise
noise = 0.8 * np.random.randn(len(t))

# Inject intermittent electrical arc spikes
spikes = np.zeros_like(t)
spike_indices = np.random.choice(len(t), size=8, replace=False)
spikes[spike_indices] = np.random.choice([-1, 1], size=8) * 6.5

raw_telemetry = clean_signal + noise + spikes
print(f"✓ Generated {len(raw_telemetry)} telemetry samples across {duration}s at {sampling_rate} Hz.")


In [ ]:
# 2. Digital Filtering & Waveform Inspection
# Apply Moving Average filter
window_size = 15
kernel = np.ones(window_size) / window_size
smoothed_ma = np.convolve(raw_telemetry, kernel, mode='same')

# Apply 4th-order Butterworth Low-Pass Filter
nyquist = 0.5 * sampling_rate
cutoff = 45.0  # Hz
b, a = signal.butter(4, cutoff / nyquist, btype='low')
smoothed_butter = signal.filtfilt(b, a, raw_telemetry)

plt.figure(figsize=(14, 5))
plt.plot(t[:300], raw_telemetry[:300], color='#ef4444', alpha=0.5, label='Raw Telemetry with Noise & Spikes')
plt.plot(t[:300], smoothed_ma[:300], color='#f59e0b', linewidth=1.8, label=f'Moving Average (N={window_size})')
plt.plot(t[:300], smoothed_butter[:300], color='#10b981', linewidth=2.2, label='Zero-Phase Butterworth (Cutoff 45Hz)')
plt.title(f"Telemetry Signal Conditioning Pipeline - {title}", fontsize=13, fontweight='bold')
plt.xlabel("Time [seconds]")
plt.ylabel("Sensor Amplitude [V / m/s²]")
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Detect slow thermal baseline wander in continuous telemetry *(🇵🇱 Wykryj powolny dryf termiczny linii bazowej w ciągłej telemetrii)*
- [ ] **Task 2:** Fit polynomial and high-pass detrending algorithms *(🇵🇱 Dopasuj algorytmy usuwania trendu (detrend wielomianowy i górnoprzepustowy))*
- [ ] **Task 3:** Implement automated zero-tare calibration logic *(🇵🇱 Zaimplementuj automatyczne zerowanie i tarowanie czujnika)*


In [ ]:
# Exercise Challenge:
# 1. Calculate the Root Mean Square (RMS) error between clean_signal and smoothed_butter.
# 2. Experiment with cutoff = 25 Hz vs 60 Hz and observe the signal attenuation.

rms_error = np.sqrt(np.mean((clean_signal - smoothed_butter)**2))
print(f"Filter Reconstruction RMS Error: {rms_error:.4f}")
assert rms_error < 1.5, "Reconstruction error is too high! Tune the filter parameters."
print("✓ Exercise verification passed.")
